In [1]:
from pathlib import Path

import astropy.units as u
import batoid
import numpy as np
import yaml
from batoid_rubin import LSSTBuilder
from astropy.coordinates import Angle

from lsst.obs.lsst import LsstCam

import StarSharp

import matplotlib.pyplot as plt
%matplotlib widget

In [2]:
telescope = batoid.Optic.fromYaml("LSST_r.yaml")
builder = LSSTBuilder(
    telescope,
    dof_coord_system="OCS",
    flip_m2_bending_modes=False,
    dof_angle_units="degree"
)
camera = LsstCam().getCamera()

In [3]:
model = StarSharp.RaytracedOpticalModel(
    builder,
    rtp = Angle("20 deg"),
    wavelength=620 * u.nm,
    camera=camera
)
field_coords = model.make_ccd_field(
    nx=4,
)

In [4]:
# Initial focus using just the camera and raft centers
sf_focus = StarSharp.StateFactory(50, use_dof="5,8,9")
guess = sf_focus.from_x([0.0]*3)
focus_field = model.make_ccd_field(nx=1, detnums=list(range(4, 189, 9)))
result = model.optimize(guess=guess, field=focus_field, nrad=5)

In [7]:
# Now make a new state factory to explore with
sf = StarSharp.StateFactory(50, use_dof="0-9,1-16,30-34")
# Let's see what happens if we tilt the camera a tiny bit
state_value = np.zeros(len(sf.use_dof))
state_value[8] = 1./3600  # 1 arcsec tilt in Camera
state = sf.from_x(state_value)

# spots = model.spots(field_coords, state=result)
spots = model.spots(field_coords, state=result + state)
moments = spots.compute_moments()

In [9]:
field_frame = "ocs"
moments_frame = "ocs"

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 4), constrained_layout=True)
axs[0].scatter(
    getattr(moments.field, field_frame).x.to_value(u.mm),
    getattr(moments.field, field_frame).y.to_value(u.mm),
    c=getattr(moments, moments_frame).T.to_value(u.mm**2),
    s=10,
    cmap="bwr",
)
axs[1].scatter(
    getattr(moments.field, field_frame).x.to_value(u.mm),
    getattr(moments.field, field_frame).y.to_value(u.mm),
    c=getattr(moments, moments_frame).e1,
    s=10,
    cmap="bwr",
)
axs[2].scatter(
    getattr(moments.field, field_frame).x.to_value(u.mm),
    getattr(moments.field, field_frame).y.to_value(u.mm),
    c=getattr(moments, moments_frame).e2,
    s=10,
    cmap="bwr",
)
for ax in axs:
    ax.set_aspect("equal")

axs[0].set_title("T")
axs[1].set_title("e1")
axs[2].set_title("e2")
plt.show()

In [10]:
# Third order moments

moments3 = spots.compute_moments(order=3)

fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(12, 3), constrained_layout=True)
axs[0].scatter(
    getattr(moments3.field, field_frame).x.to_value(u.mm),
    getattr(moments3.field, field_frame).y.to_value(u.mm),
    c=getattr(moments3, moments_frame).spin(3).value,
    s=7,
    cmap="bwr",
)
axs[1].scatter(
    getattr(moments3.field, field_frame).x.to_value(u.mm),
    getattr(moments3.field, field_frame).y.to_value(u.mm),
    c=getattr(moments3, moments_frame).spin(1).value,
    s=7,
    cmap="bwr",
)
axs[2].scatter(
    getattr(moments3.field, field_frame).x.to_value(u.mm),
    getattr(moments3.field, field_frame).y.to_value(u.mm),
    c=getattr(moments3, moments_frame).spin(-1).value,
    s=7,
    cmap="bwr",
)
axs[3].scatter(
    getattr(moments3.field, field_frame).x.to_value(u.mm),
    getattr(moments3.field, field_frame).y.to_value(u.mm),
    c=getattr(moments3, moments_frame).spin(-3).value,
    s=7,
    cmap="bwr",
)
for ax in axs:
    ax.set_aspect("equal")
axs[0].set_title("spin 3")
axs[1].set_title("spin 1")
axs[2].set_title("spin -1")
axs[3].set_title("spin -3")
plt.show()